# Real-time Fraud Dashboard (TP5 §5.3)

## Run order

1. Terminal: `./scripts/test.sh --full` → wait for **SUCCESS**
2. Browser: http://127.0.0.1:8888 (token: `ChangeMeStrong`)
3. Open this notebook → **Run → Run All Cells**

Docker must still be running (`jupyter` container). Data is read from `/workspace/data/metrics/` (same files as `python scripts/show_metrics.py`).

## Assignment mapping

| Requirement | Where |
|-------------|--------|
| §5.2 Kafka consume + JSON parse | `spark_jobs/fraud_metrics_stream.py` |
| Sliding windows 3h / 7d / 3w / 3mo | `spark_jobs/config.py` `WINDOW_SPECS` |
| Per-user stats + distinct peers | `avg_amount`, `tx_count`, `distinct_peers` in snapshot |
| Global state (lifetime) | `data/metrics/lifetime/sent`, `.../received` |
| Checkpointing + watermark | `checkpoints/fraud-metrics/`, 10 min watermark |
| §5.3 Last 20 users | Section below |
| §5.3 Last 10s activity | Section below |
| §5.3 Auto-refresh | Last cell (5 s) |

In [1]:
%matplotlib inline
import os
from datetime import timedelta, timezone
from datetime import datetime as dt

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

METRICS_DIR = os.environ.get("METRICS_DIR", "/workspace/data/metrics")
REFRESH_SECONDS = int(os.environ.get("DASHBOARD_REFRESH", "5"))

print("Metrics dir:", METRICS_DIR)
if os.path.isdir(METRICS_DIR):
    print("Files:", os.listdir(METRICS_DIR))
else:
    print("Missing — run ./scripts/test.sh --full first")

Metrics dir: /workspace/data/metrics
Files: ['.gitkeep', 'lifetime', 'latest_snapshot.parquet', 'batches', 'recent_transactions.parquet']


In [2]:
def load_parquet(path):
    if not os.path.exists(path):
        return pd.DataFrame()
    return pd.read_parquet(path)


def _filter_test_users(df, col="user_id"):
    if df.empty or col not in df.columns:
        return df
    return df[~df[col].astype(str).str.startswith("__")]


def style_alerts(df):
    if df.empty:
        return df
    return df.style.apply(lambda _: ["background-color: #ffcccc"] * len(df.columns), axis=1)


def render_dashboard():
    recent = load_parquet(os.path.join(METRICS_DIR, "recent_transactions.parquet"))
    metrics = load_parquet(os.path.join(METRICS_DIR, "latest_snapshot.parquet"))
    alerts = load_parquet(os.path.join(METRICS_DIR, "alerts.parquet"))
    lifetime_sent = load_parquet(os.path.join(METRICS_DIR, "lifetime", "sent"))

    print(f"=== Dashboard @ {dt.now(timezone.utc).isoformat()} ===")
    ready = os.path.exists(os.path.join(METRICS_DIR, "recent_transactions.parquet"))
    print(f"Data ready: {ready}")
    if not ready:
        print("Run: ./scripts/test.sh --full")
        return

    # --- Last 10 seconds of activity (§5.3) ---
    print("\n## Last 10 seconds of activity")
    if recent.empty:
        print("No recent transactions.")
    else:
        r = recent.copy()
        if "event_time" in r.columns:
            r["event_time"] = pd.to_datetime(r["event_time"], utc=True)
            tmax = r["event_time"].max()
            last10 = r[r["event_time"] >= tmax - timedelta(seconds=10)]
            print(f"Window: {tmax - timedelta(seconds=10)} → {tmax} ({len(last10)} txs)")
        elif "date" in r.columns:
            r["date"] = pd.to_datetime(r["date"], utc=True)
            tmax = r["date"].max()
            last10 = r[r["date"] >= tmax - timedelta(seconds=10)]
            print(f"Window: last 10s of simulated time ({len(last10)} txs)")
        else:
            last10 = r.tail(50)
        cols = [c for c in ["send_id", "receive_id", "amount", "date", "event_time", "sim_fraud"] if c in last10.columns]
        display(last10[cols].tail(30))

    # --- Last 20 active users (§5.3) ---
    print("\n## Last 20 users (most recent senders)")
    if recent.empty or "send_id" not in recent.columns:
        print("No user data.")
    else:
        sort_col = "event_time" if "event_time" in recent.columns else "date"
        r = recent.copy()
        if sort_col in r.columns:
            r[sort_col] = pd.to_datetime(r[sort_col], utc=True)
        last20 = (
            r.sort_values(sort_col)
            .drop_duplicates(subset=["send_id"], keep="last")
            .tail(20)
        )
        cols = [c for c in ["send_id", "receive_id", "amount", sort_col, "sim_fraud"] if c in last20.columns]
        display(last20[cols])

    # --- Windowed metrics: avg, count, distinct peers (§5.3) ---
    print("\n## Windowed metrics (avg_amount, tx_count, distinct_peers)")
    metrics = _filter_test_users(metrics)
    if metrics.empty:
        print("No snapshot.")
    else:
        show = [c for c in [
            "user_id", "direction", "window_name", "window_start", "window_end",
            "avg_amount", "tx_count", "distinct_peers", "computed_at"
        ] if c in metrics.columns]
        top = metrics.sort_values("tx_count", ascending=False).head(25)[show]
        display(top)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        metrics.groupby("window_name")["tx_count"].sum().sort_values(ascending=False).plot(
            kind="bar", ax=axes[0], color="steelblue"
        )
        axes[0].set_title("Total tx_count by window")
        metrics.groupby("window_name")["distinct_peers"].mean().plot(
            kind="bar", ax=axes[1], color="coral"
        )
        axes[1].set_title("Mean distinct_peers by window")
        plt.tight_layout()
        plt.show()

    # --- Lifetime / global state (§5.2 #4) ---
    print("\n## Lifetime metrics (since start)")
    lt = _filter_test_users(lifetime_sent)
    if lt.empty:
        print("No lifetime/sent data yet.")
    else:
        cols = [c for c in ["user_id", "tx_count", "total_amount", "distinct_peers"] if c in lt.columns]
        display(lt.sort_values("tx_count", ascending=False).head(15)[cols])

    # --- Alerts (red highlight) ---
    print("\n## Fraud alerts")
    if alerts.empty:
        print("No alerts in this run.")
    else:
        display(style_alerts(alerts.head(20)))

In [3]:
# Static view (run once)
render_dashboard()

=== Dashboard @ 2026-05-21T15:11:26.648939+00:00 ===
Data ready: True

## Last 10 seconds of activity
Window: 2026-05-21 15:09:43+00:00 → 2026-05-21 15:09:53+00:00 (149 txs)


,send_id,receive_id,amount,date,event_time,sim_fraud
937,U_000157,U_000001,3.99,2026-05-21T15:09:44Z,2026-05-21 15:09:44+00:00,None
938,U_000092,U_000021,4.61,2026-05-21T15:09:45Z,2026-05-21 15:09:45+00:00,None
939,U_000101,U_000119,1.34,2026-05-21T15:09:45Z,2026-05-21 15:09:45+00:00,None
940,U_000157,U_000195,6.48,2026-05-21T15:09:45Z,2026-05-21 15:09:45+00:00,None
941,U_000188,U_000040,15.09,2026-05-21T15:09:45Z,2026-05-21 15:09:45+00:00,None
942,U_000098,U_000031,4.05,2026-05-21T15:09:46Z,2026-05-21 15:09:46+00:00,None
943,U_000160,U_000059,16.16,2026-05-21T15:09:46Z,2026-05-21 15:09:46+00:00,None
944,U_000006,U_000132,8.35,2026-05-21T15:09:47Z,2026-05-21 15:09:47+00:00,None
945,U_000026,U_000186,7.62,2026-05-21T15:09:47Z,2026-05-21 15:09:47+00:00,None
946,U_000046,U_000051,119.21,2026-05-21T15:09:47Z,2026-05-21 15:09:47+00:00,None



## Last 20 users (most recent senders)


,send_id,receive_id,amount,event_time,sim_fraud
325,U_000017,U_000194,8.51,2026-05-21 15:09:52+00:00,None
326,U_000179,U_000046,1.76,2026-05-21 15:09:52+00:00,None
327,U_000199,U_000148,5.99,2026-05-21 15:09:52+00:00,True
479,U_000197,U_000054,86.87,2026-05-21 15:09:52+00:00,None
960,U_000006,U_000052,9.22,2026-05-21 15:09:52+00:00,None
780,U_000190,U_000094,1.07,2026-05-21 15:09:52+00:00,None
962,U_000073,U_000147,3.95,2026-05-21 15:09:53+00:00,None
963,U_000090,U_000084,2.16,2026-05-21 15:09:53+00:00,None
964,U_000155,U_000156,59.23,2026-05-21 15:09:53+00:00,None
483,U_000169,U_000018,23.03,2026-05-21 15:09:53+00:00,None



## Windowed metrics (avg_amount, tx_count, distinct_peers)
No snapshot.

## Lifetime metrics (since start)
No lifetime/sent data yet.

## Fraud alerts
No alerts in this run.


In [4]:
# Auto-refresh every 5 seconds (§5.3) — Interrupt kernel to stop
while True:
    clear_output(wait=True)
    render_dashboard()
    print(f"\nRefreshing in {REFRESH_SECONDS}s... (Interrupt kernel to stop)")
    import time
    time.sleep(REFRESH_SECONDS)

=== Dashboard @ 2026-05-21T15:21:09.568464+00:00 ===
Data ready: True

## Last 10 seconds of activity
Window: 2026-05-21 15:09:43+00:00 → 2026-05-21 15:09:53+00:00 (149 txs)


,send_id,receive_id,amount,date,event_time,sim_fraud
937,U_000157,U_000001,3.99,2026-05-21T15:09:44Z,2026-05-21 15:09:44+00:00,None
938,U_000092,U_000021,4.61,2026-05-21T15:09:45Z,2026-05-21 15:09:45+00:00,None
939,U_000101,U_000119,1.34,2026-05-21T15:09:45Z,2026-05-21 15:09:45+00:00,None
940,U_000157,U_000195,6.48,2026-05-21T15:09:45Z,2026-05-21 15:09:45+00:00,None
941,U_000188,U_000040,15.09,2026-05-21T15:09:45Z,2026-05-21 15:09:45+00:00,None
942,U_000098,U_000031,4.05,2026-05-21T15:09:46Z,2026-05-21 15:09:46+00:00,None
943,U_000160,U_000059,16.16,2026-05-21T15:09:46Z,2026-05-21 15:09:46+00:00,None
944,U_000006,U_000132,8.35,2026-05-21T15:09:47Z,2026-05-21 15:09:47+00:00,None
945,U_000026,U_000186,7.62,2026-05-21T15:09:47Z,2026-05-21 15:09:47+00:00,None
946,U_000046,U_000051,119.21,2026-05-21T15:09:47Z,2026-05-21 15:09:47+00:00,None



## Last 20 users (most recent senders)


,send_id,receive_id,amount,event_time,sim_fraud
325,U_000017,U_000194,8.51,2026-05-21 15:09:52+00:00,None
326,U_000179,U_000046,1.76,2026-05-21 15:09:52+00:00,None
327,U_000199,U_000148,5.99,2026-05-21 15:09:52+00:00,True
479,U_000197,U_000054,86.87,2026-05-21 15:09:52+00:00,None
960,U_000006,U_000052,9.22,2026-05-21 15:09:52+00:00,None
780,U_000190,U_000094,1.07,2026-05-21 15:09:52+00:00,None
962,U_000073,U_000147,3.95,2026-05-21 15:09:53+00:00,None
963,U_000090,U_000084,2.16,2026-05-21 15:09:53+00:00,None
964,U_000155,U_000156,59.23,2026-05-21 15:09:53+00:00,None
483,U_000169,U_000018,23.03,2026-05-21 15:09:53+00:00,None



## Windowed metrics (avg_amount, tx_count, distinct_peers)
No snapshot.

## Lifetime metrics (since start)
No lifetime/sent data yet.

## Fraud alerts
No alerts in this run.

Refreshing in 5s... (Interrupt kernel to stop)


KeyboardInterrupt: 